# Synthesis Strategist - local LLM Resume Optimizer

This notebook demonstrates how we prompt a local conversational model (`Gemma-2-2B-it` or `Phi-3-mini-4k-instruct`) to perform alignment synthesis. It cross-references extracted job requirements with matched portfolio items and generates Google X-Y-Z formula resume bullets.

In [1]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

C:\Users\Vansh Agrawal\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


C:\Users\Vansh Agrawal\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


## 1. Prompt Engineering for Synthesis & Google X-Y-Z Formula

We formulate a system instructions prompt that guides the model to act as an executive resume optimizer. The Google X-Y-Z formula requires:
- **Accomplished [X]**, as measured by **[Y]**, by doing **[Z]**.

In [2]:
system_prompt = """
You are a professional resume writer. Review the target requirements and matching projects, then generate 2 high-impact resume bullet points aligned to the requirements.
Use the strict Google X-Y-Z formula: 'Accomplished [X], as measured by [Y], by doing [Z]'. Use quantitative metrics where possible.
"""

user_input = """
Target Job Requirements:
- Skills: Python, FastAPI, Postgres, high-throughput pipelines.

Candidate Project:
- Title: VinoMetrix
- Tech: FastAPI, XGBoost, python.
- Accomplishments: Wrote a prediction backend. Speeds up data parsing. Got 95% classification accuracy on chemical data.
"""

# Format prompt using Chat templates (e.g. ChatML style)
prompt = f"<|system|>\n{system_prompt}\n<|user|>\n{user_input}\n<|assistant|>\n"

## 2. Load Model and Generate Tailored Resume Bullets

We demonstrate running the generation pipeline locally.

In [3]:
# Example loader pointing to Phi-3-mini or Gemma-2-2B. 
# Note: When running on standard CPU, we configure parameters like device_map='auto' or load_in_8bit to save memory.
model_id = "microsoft/Phi-3-mini-4k-instruct"
try:
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id, 
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto"
    )
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=250, temperature=0.2)
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print("\n--- Generated Output ---\n")
    print(result.replace(prompt, ""))
except Exception as e:
    print(f"Skipping actual model load during demo. Exception details: {e}")
    print("To execute on local machine: run `pip install accelerate` and make sure you have enough VRAM.")

`torch_dtype` is deprecated! Use `dtype` instead!


Skipping actual model load during demo. Exception details: Using a `device_map`, `tp_plan`, `torch.device` context manager or setting `torch.set_default_device(device)` requires `accelerate`. You can install it with `pip install accelerate`
To execute on local machine: run `pip install accelerate` and make sure you have enough VRAM.
